# Molecular Hamiltonians

Turn a hydrogen molecule into a four-qubit operator and read its ground-state energy straight off the spectrum.

**Objectives:**
- Inspect the real STO-3G H2 Jordan-Wigner Hamiltonian as a weighted sum of Pauli strings
- Measure its structure: number of terms, maximum Pauli locality, and the identity/coefficient breakdown
- Build the dense matrix and diagonalize it to recover the exact ground energy, confirming it equals the FCI value
- Evaluate the Hartree-Fock determinant |1100> and see why it sits *above* the true ground state
- Read (but not run) the classical pipeline that produced these coefficients: geometry -> integrals -> Jordan-Wigner

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Browser-runnable: this notebook executes in-browser against qcsim,
# a pure-NumPy Braket stand-in. No AWS or pip install required.

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector

# Deterministic: tight asserts below rely on exact eigenvalues / state vectors,
# never on raw sampling, but we seed anyway for reproducibility.
np.random.seed(0)

# qcsim is BIG-ENDIAN: qubit 0 is the leftmost (most-significant) bit of a ket,
# so the basis state |1100> means qubits 0 and 1 are occupied.
device = LocalSimulator()

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. The molecule, already an operator

The setup above injected a verified **chemistry kit**. The star of this notebook is `H2_TERMS`: the exact electronic-structure Hamiltonian for a hydrogen molecule in the minimal STO-3G basis at bond length R = 0.75 Angstrom, after the Jordan-Wigner transformation has rewritten it for qubits.

It is a Python list of `(pauli_string, coeff)` pairs. Each `pauli_string` is four characters drawn from `I`, `X`, `Y`, `Z`; character `k` is the Pauli operator acting on qubit `k` (big-endian, so character 0 acts on qubit 0). The whole Hamiltonian is just the weighted sum

$$H = \sum_i c_i \, P_i .$$

No approximation has been made here. This *is* the H2 problem, simply written in a language a four-qubit device can measure. Let us look at it.

In [ ]:
# Print every Pauli term and its coefficient.
print(f"H2_TERMS holds {len(H2_TERMS)} Pauli terms on {len(H2_TERMS[0][0])} qubits\n")
print(f"{'qubits 0123':>12}   coefficient (Ha)")
print("-" * 33)
for pauli, coeff in H2_TERMS:
    print(f"{pauli:>12}   {coeff:+.6f}")

# The kit also exposes the reference energies we will check against.
print(f"\nH2_FCI (exact ground energy) = {H2_FCI:.6f} Ha")
print(f"H2_HF  (Hartree-Fock |1100>) = {H2_HF:.6f} Ha")

## 2. Reading the structure

Three numbers describe the *shape* of a qubit Hamiltonian, and they decide how expensive it is to work with on real hardware:

- **Number of terms** — how many Pauli strings the Hamiltonian is a sum of. For STO-3G H2 this is **15**. That is a term count, not a measurement budget: the identity string below is a free classical constant, so a naive VQE measures the other **14** as separate hardware tasks, and grouping commuting strings shrinks that to about **five** settings.
- **Maximum Pauli locality** — the length of the longest non-identity string, i.e. how many qubits the most entangling term touches. The Jordan-Wigner Z-strings push H2's worst terms (the `XXYY`-type doubles) onto all **4** qubits.
- **The identity coefficient** — the `IIII` term is a constant energy shift. It moves every eigenvalue uniformly and needs no quantum measurement at all.

We compute the locality of a term by counting its non-`I` characters.

In [ ]:
def term_locality(pauli):
    """Number of qubits a Pauli string acts on non-trivially (its weight)."""
    return sum(1 for ch in pauli if ch != "I")

n_terms = len(H2_TERMS)
max_locality = max(term_locality(p) for p, _ in H2_TERMS)

# Split the identity (constant) part from the rest.
identity_coeff = sum(c for p, c in H2_TERMS if term_locality(p) == 0)

# Tally how many terms act on each number of qubits.
from collections import Counter
locality_hist = Counter(term_locality(p) for p, _ in H2_TERMS)

print(f"number of terms      : {n_terms}")
print(f"max Pauli locality   : {max_locality}")
print(f"identity coefficient : {identity_coeff:+.6f} Ha (constant shift)")
print("\nterms by locality (k = qubits touched):")
for k in sorted(locality_hist):
    label = "identity" if k == 0 else f"{k}-local"
    print(f"  k={k} ({label:>8}): {locality_hist[k]} terms")

# The full-weight terms are the doubles that act on ALL four qubits.
full_weight = [p for p, _ in H2_TERMS if term_locality(p) == 4]
print(f"\nterms touching all 4 qubits: {full_weight}")

assert n_terms == 15, "STO-3G H2 must have exactly 15 Pauli terms"
assert max_locality == 4, "the XXYY/XYYX-type doubles act on all 4 qubits"
print("\nOK: 15 terms, max locality 4.")

## 3. From terms to a matrix

A Pauli string is shorthand for a $2^n \times 2^n$ matrix built by tensoring the single-qubit Paulis in order. Because we are big-endian, the leftmost character occupies the *outermost* (most significant) factor of the Kronecker product:

$$P = P_{q_0} \otimes P_{q_1} \otimes \cdots \otimes P_{q_{n-1}} .$$

The kit gives us two helpers so we never have to assemble these by hand:

- `pauli_matrix(s)` — the dense matrix of one big-endian Pauli string `s`.
- `hamiltonian_matrix(terms)` — the full $16 \times 16$ Hermitian Hamiltonian $\sum_i c_i P_i$ from a list of `(string, coeff)` pairs.

Let us build the matrix and sanity-check that it is Hermitian (energies are real, so $H = H^\dagger$).

In [ ]:
# A single Pauli string -> its 16x16 matrix.
ZIII = pauli_matrix("ZIII")
print(f"pauli_matrix('ZIII') has shape {ZIII.shape}")

# The full molecular Hamiltonian as a dense matrix.
H = hamiltonian_matrix(H2_TERMS)
print(f"hamiltonian_matrix(H2_TERMS) has shape {H.shape}")

# H must be Hermitian: a physical observable has real eigenvalues.
hermitian_error = np.max(np.abs(H - H.conj().T))
print(f"max |H - H^dagger| = {hermitian_error:.2e}")
assert hermitian_error < 1e-12, "the Hamiltonian must be Hermitian"
print("OK: H is Hermitian.")

## 4. Diagonalize: the exact ground energy

The ground-state energy of the molecule is the **lowest eigenvalue** of $H$. For a four-qubit Hamiltonian the matrix is only $16 \times 16$, so we can diagonalize it directly with `np.linalg.eigvalsh` (the `h` is for *Hermitian* — it returns sorted real eigenvalues).

This brute-force diagonalization is exactly the exponential wall VQE is built to avoid: it works beautifully for H2 but doubles in cost with every spin-orbital. Here it gives us the ground truth to check VQE against. The lowest eigenvalue should equal the kit's `H2_FCI`, the full-configuration-interaction (exact) energy.

In [ ]:
eigenvalues = np.linalg.eigvalsh(hamiltonian_matrix(H2_TERMS))
ground_energy = eigenvalues[0]

print("lowest four eigenvalues (Ha):")
for i, e in enumerate(eigenvalues[:4]):
    print(f"  E_{i} = {e:+.6f}")

print(f"\nground energy   = {ground_energy:.6f} Ha")
print(f"H2_FCI (kit)    = {H2_FCI:.6f} Ha")
print(f"difference      = {abs(ground_energy - H2_FCI):.2e}")

assert np.isclose(np.linalg.eigvalsh(hamiltonian_matrix(H2_TERMS))[0], H2_FCI, atol=1e-4)
print("\nOK: the lowest eigenvalue of H2_TERMS equals H2_FCI.")

## 5. The Hartree-Fock determinant |1100>

Before any quantum optimization, chemistry hands us a first guess: the **Hartree-Fock** state, the single best mean-field determinant. For H2 that determinant fills the two lowest spin-orbitals, which in our big-endian convention is the computational basis state

$$\ket{\text{HF}} = \ket{1100} \quad\text{(qubits 0 and 1 occupied, qubits 2 and 3 empty).}$$

The energy of any state $\ket{\psi}$ under the Hamiltonian is the expectation value $\langle\psi|H|\psi\rangle$. The kit's `hamiltonian_energy(state_vector, terms)` computes exactly that. We build the 16-component basis vector for `|1100>` by placing a 1 at index `int("1100", 2)` — the big-endian integer the bitstring names — and ask for its energy. It must match the kit's `H2_HF`.

Crucially, $E_{\text{HF}} > E_{\text{ground}}$: the mean-field determinant is a valid trial state, so by the variational principle it can only sit *above* the true floor. The gap is the **correlation energy** that VQE must recover.

In [ ]:
# Big-endian basis state |1100>: a single 1 at the integer index named by the bits.
hf_index = int("1100", 2)  # = 12
hf_state = np.zeros(2 ** 4, dtype=complex)
hf_state[hf_index] = 1.0
print(f"|1100> -> basis index {hf_index} of {hf_state.size}")

hf_energy = hamiltonian_energy(hf_state, H2_TERMS)
print(f"\nHartree-Fock energy <1100|H|1100> = {hf_energy:.6f} Ha")
print(f"H2_HF (kit)                       = {H2_HF:.6f} Ha")

correlation_energy = ground_energy - hf_energy
print(f"\nground energy (FCI)   = {ground_energy:.6f} Ha")
print(f"correlation energy    = {correlation_energy:.6f} Ha  (FCI - HF, negative)")

assert np.isclose(hamiltonian_energy(hf_state, H2_TERMS), H2_HF, atol=1e-4)
assert hf_energy > H2_FCI, "Hartree-Fock must lie above the true ground state"
print("\nOK: |1100> energy equals H2_HF, and it sits above the FCI ground energy.")

## 6. Reaching the ground state with one excitation

Hartree-Fock misses the ground state because it cannot mix in the doubly-excited determinant `|0011>` (both electrons promoted). A single **Givens double excitation** rotates `|1100>` toward `|0011>` by an angle $\theta$, and at the optimal angle it lands *exactly* on the H2 ground state.

The kit packages this as `h2_double_excitation(theta)`: it prepares `|1100>` and applies that one excitation. We can scan $\theta$, evaluate the energy of each resulting state with `hamiltonian_energy`, and watch the curve dip from the Hartree-Fock value down to the FCI floor. This is the entire VQE landscape for H2 in a single parameter.

In [ ]:
thetas = np.linspace(-np.pi, np.pi, 121)
energies = []
for theta in thetas:
    circ = h2_double_excitation(theta)
    # statevector(circ) spans all 4 qubits here, matching the 16x16 Hamiltonian.
    psi = np.asarray(statevector(circ))
    energies.append(hamiltonian_energy(psi, H2_TERMS))
energies = np.array(energies)

best_i = int(np.argmin(energies))
best_theta, best_energy = thetas[best_i], energies[best_i]
print(f"best theta on grid = {best_theta:+.4f} rad")
print(f"energy there       = {best_energy:.6f} Ha")
print(f"H2_FCI             = {H2_FCI:.6f} Ha")

# The single excitation reaches the exact ground state at its optimum.
assert np.isclose(best_energy, H2_FCI, atol=1e-3)
print("\nOK: the double excitation reaches the FCI ground energy.")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thetas, energies, color="#2563eb", lw=2, label=r"$E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle$")
ax.axhline(H2_FCI, color="#dc2626", ls="--", lw=1.5, label=f"FCI floor = {H2_FCI:.4f} Ha")
ax.axhline(H2_HF, color="#6b7280", ls=":", lw=1.5, label=f"Hartree-Fock = {H2_HF:.4f} Ha")
ax.scatter([best_theta], [best_energy], color="#dc2626", zorder=5, s=40)
ax.set_xlabel(r"excitation angle $\theta$ (rad)")
ax.set_ylabel("energy (Hartree)")
ax.set_title("H2 energy vs. double-excitation angle")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

## 7. The same molecule on a single qubit

Those four qubits carry redundancy. The conserved quantities (electron number and spin parity) mean the real H2 problem is far smaller than four qubits suggests. **Tapering** away the symmetries collapses the fifteen-term, four-qubit operator down to a single qubit with just three terms:

$$H_{\text{H}_2} \approx c_0\, I + c_z\, Z + c_x\, X .$$

The kit supplies these tapered coefficients as `H2_C0`, `H2_CZ`, `H2_CX`. The ground energy of a single-qubit $c_0 I + c_z Z + c_x X$ has a closed form, $c_0 - \sqrt{c_z^2 + c_x^2}$ — and it lands on the very same `H2_FCI`. One molecule, two faithful descriptions; the lesson is that the naive qubit count is almost never the real one.

In [ ]:
# Closed-form ground energy of c0*I + cz*Z + cx*X.
tapered_ground = H2_C0 - np.hypot(H2_CZ, H2_CX)
print(f"tapered Hamiltonian:  H = {H2_C0:+.4f} I {H2_CZ:+.4f} Z {H2_CX:+.4f} X")
print(f"ground = c0 - hypot(cz, cx) = {tapered_ground:.6f} Ha")
print(f"H2_FCI                       = {H2_FCI:.6f} Ha")

# Cross-check by diagonalizing the 2x2 tapered matrix directly.
H_tapered = hamiltonian_matrix([("I", H2_C0), ("Z", H2_CZ), ("X", H2_CX)])
tapered_eigs = np.linalg.eigvalsh(H_tapered)
print(f"diagonalized 2x2 lowest eigenvalue = {tapered_eigs[0]:.6f} Ha")

assert np.isclose(tapered_ground, H2_FCI, atol=1e-4)
assert np.isclose(tapered_eigs[0], H2_FCI, atol=1e-4)
print("\nOK: the one-qubit tapered Hamiltonian gives the same ground energy as the 4-qubit one.")

## 8. Reference: where these coefficients come from

The coefficients in `H2_TERMS` are not magic — a classical computer grinds them out once from the nuclear geometry. The pipeline is **geometry -> molecular integrals -> fermionic operator -> Jordan-Wigner -> qubit Hamiltonian**, and it lives in `lib/chemistry/hamiltonians.py` as `build_h2_hamiltonian`.

That pipeline needs the full OpenFermion + PySCF stack, which does **not** run in this browser (Pyodide) environment, so the cell below is shown for reference only and is intentionally *not executed here*. Run it in the full `[dev]` environment (`pip install -e '.[full]'`, then `make lab`) to regenerate the very `H2_TERMS` this notebook consumed.

```python
# REFERENCE ONLY -- requires openfermion + openfermionpyscf + pyscf.
# Do NOT run in-browser; this is the classical pre-processing that
# produced the H2_TERMS the kit injected above.
from openfermion.chem import MolecularData
from openfermionpyscf import run_pyscf
from openfermion.transforms import get_fermion_operator, jordan_wigner

# 1. Geometry: two H atoms, 0.75 Angstrom apart, minimal STO-3G basis.
geometry = [("H", (0.0, 0.0, 0.0)), ("H", (0.0, 0.0, 0.75))]
molecule = MolecularData(geometry, "sto-3g", multiplicity=1, charge=0)

# 2. Integrals: PySCF computes the one- and two-electron integrals
#    h_pq and h_pqrs (pure geometry + basis set), plus the FCI energy.
molecule = run_pyscf(molecule, run_fci=True)

# 3. Fermionic operator: fold the integrals into sum_pq h_pq a_p^dag a_q + ...
fermion_op = get_fermion_operator(molecule.get_molecular_hamiltonian())

# 4. Jordan-Wigner: rewrite a_p^dag / a_p as Pauli strings with Z-strings,
#    yielding the 15-term, 4-qubit qubit Hamiltonian.
qubit_hamiltonian = jordan_wigner(fermion_op)

# qubit_hamiltonian.terms is the dict equivalent of H2_TERMS;
# molecule.fci_energy is H2_FCI. The convenience wrapper is:
#   from lib.chemistry.hamiltonians import build_h2_hamiltonian
#   qubit_hamiltonian, n_qubits, n_electrons = build_h2_hamiltonian(0.75)
```

## Exercises

Four exercises to cement the mechanics: ranking the terms by weight, truncating
the Hamiltonian, scoring single determinants, and testing the variational
floor. Each one gives you a prompt with two tiered hints -- open only what you
need -- a scaffold cell to write your attempt in, and a check cell that tells
you when you have it. They reuse only the kit names (`H2_TERMS`,
`hamiltonian_matrix`, `hamiltonian_energy`, `H2_FCI`) and the allowed imports.

### Exercise 1 — Coefficient census

Which Pauli terms carry the most energy? Rank `H2_TERMS` by the magnitude of
its coefficients and pull out the heaviest few.

Define `top3_terms`: a list of the three `(pauli_string, coeff)` pairs from
`H2_TERMS` with the largest `abs(coeff)`, ordered from largest magnitude to
smallest. Print them, and notice whether the dominant terms are the diagonal
`Z`-only strings or the off-diagonal doubles.

<details><summary>Hint 1 — nudge</summary>

Each entry is a `(pauli_string, coeff)` pair, and "heaviest" means largest
*magnitude* -- a big negative coefficient counts just as much as a big positive
one. Which one-argument function turns "size regardless of sign" into a number?

</details>
<details><summary>Hint 2 — approach</summary>

Reach for `sorted(H2_TERMS, key=..., reverse=True)` with a key that returns
`abs(coeff)` for each pair, then slice the first three. `term_locality` from
Section 2 will confirm whether the survivors are the `Z`-only diagonal terms.

</details>

In [ ]:
# Exercise 1: Rank H2_TERMS by coefficient magnitude and keep the heaviest three.
# Define: top3_terms -- list of the 3 (pauli_string, coeff) pairs with the
# largest abs(coeff), ordered largest magnitude first.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert len(top3_terms) == 3, "keep exactly the three heaviest terms"
    assert all(pair in H2_TERMS for pair in top3_terms), (
        "each entry must be an unmodified (pauli_string, coeff) pair from H2_TERMS"
    )
    mags = [abs(c) for _, c in top3_terms]
    assert mags == sorted(mags, reverse=True), (
        "order the three from largest magnitude down to smallest"
    )
    _top_paulis = {p for p, _ in top3_terms}
    _rest = [abs(c) for p, c in H2_TERMS if p not in _top_paulis]
    assert min(mags) >= max(_rest), (
        "these must be the three LARGEST magnitudes -- no lighter term outranks them"
    )

### Exercise 2 — Drop the small terms

How much do the tiny coefficients actually matter? Throw away every term whose
coefficient is small and see what it costs the ground energy.

Define `kept_terms`: the sublist of `H2_TERMS` whose coefficients satisfy
`abs(coeff) > 0.05`. Then define `trunc_ground`: the lowest eigenvalue of the
Hamiltonian built from `kept_terms` alone. Compare it to `H2_FCI` -- how many
milli-Hartree did the discarded terms buy?

<details><summary>Hint 1 — nudge</summary>

Section 3 built the full matrix from a list of terms and Section 4 read the
ground energy off its spectrum. This is those same two steps run on a *shorter*
list of terms.

</details>
<details><summary>Hint 2 — approach</summary>

Filter with a comprehension: `[(p, c) for p, c in H2_TERMS if abs(c) > 0.05]`.
Feed that shorter list to `hamiltonian_matrix(...)`, hand the matrix to
`np.linalg.eigvalsh(...)`, and take element `[0]` -- the smallest eigenvalue,
exactly as Section 4 did for the full operator.

</details>

In [ ]:
# Exercise 2: Truncate H2_TERMS to its large coefficients and diagonalize.
# Define: kept_terms -- terms with abs(coeff) > 0.05; trunc_ground -- the
# lowest eigenvalue of the Hamiltonian built from kept_terms.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert all(abs(c) > 0.05 for _, c in kept_terms), (
        "kept_terms should hold only terms with abs(coeff) > 0.05"
    )
    assert 0 < len(kept_terms) < len(H2_TERMS), (
        "truncation should drop some terms but not all of them"
    )
    assert all(pair in H2_TERMS for pair in kept_terms), (
        "keep the original (pauli_string, coeff) pairs unchanged"
    )
    _tg = float(np.real(trunc_ground))
    assert _tg > H2_FCI + 1e-4, (
        "here dropping the small off-diagonal terms should raise the floor: "
        "trunc_ground sits ABOVE H2_FCI"
    )
    assert _tg - H2_FCI < 0.1, (
        "the discarded terms are small -- the truncated ground energy stays within "
        "a few tens of milli-Hartree of the exact one"
    )

### Exercise 3 — Other determinants

The Hartree-Fock guess was `|1100>`. What energy would other single
determinants give? Score three of them and find the best.

Define `det_energies`: a dict mapping each bitstring `"1010"`, `"0110"`,
`"0011"` to its energy `hamiltonian_energy(v, H2_TERMS)`, where `v` is the
big-endian basis vector for that bitstring. Which determinant lands lowest, and
why is even the best one still above the true ground state?

<details><summary>Hint 1 — nudge</summary>

Section 5 built one basis vector, `|1100>`, by placing a single 1 at the
integer index its bits name, then handed it to `hamiltonian_energy`. Each
determinant here is that same construction with different bits.

</details>
<details><summary>Hint 2 — approach</summary>

Loop over the three bitstrings. For each: make a length-16 complex zero vector,
set index `int(bits, 2)` to 1.0, and store `hamiltonian_energy(v, H2_TERMS)`
under the bitstring key. Every determinant is a valid trial state, so the
variational principle keeps each energy at or above `H2_FCI`.

</details>

In [ ]:
# Exercise 3: Energies of three single determinants.
# Define: det_energies -- dict {bits: hamiltonian_energy(v, H2_TERMS)} for
# bits in "1010", "0110", "0011", with v the big-endian basis vector for bits.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert set(det_energies) == {"1010", "0110", "0011"}, (
        "record exactly the three determinants named in the prompt"
    )
    assert all(abs(np.imag(e)) < 1e-9 for e in det_energies.values()), (
        "an energy <psi|H|psi> of a Hermitian H is real"
    )
    assert all(np.real(e) >= H2_FCI - 1e-9 for e in det_energies.values()), (
        "every determinant is a trial state -- none can dip below H2_FCI"
    )
    assert min(np.real(e) for e in det_energies.values()) > H2_FCI + 1e-3, (
        "no single determinant is the ground state; the best one is still strictly above it"
    )

### Exercise 4 — Verify the variational floor

The variational principle says *no* normalized state can have an energy below
the ground energy. Test it empirically with random states.

Define `random_energies`: a list of `hamiltonian_energy(v, H2_TERMS)` for 20
random normalized 16-dimensional complex state vectors. Confirm every one lands
at or above `H2_FCI`. Seed with `np.random.seed(0)` so your run is reproducible.

<details><summary>Hint 1 — nudge</summary>

A random state is still a valid state, so the floor from Section 5 -- the
variational principle -- must hold for all 20. The bound is on the *energy*, not
on how the vector happened to be drawn.

</details>
<details><summary>Hint 2 — approach</summary>

Loop 20 times. Each pass: draw `np.random.randn(16) + 1j * np.random.randn(16)`,
divide by its `np.linalg.norm` to normalize, and append
`hamiltonian_energy(v, H2_TERMS)`. Collect the 20 numbers into `random_energies`.

</details>

In [ ]:
# Exercise 4: Empirically confirm the variational floor with random states.
# Define: random_energies -- list of hamiltonian_energy(v, H2_TERMS) for 20
# random normalized 16-dimensional complex vectors (np.random.seed(0)).

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert len(random_energies) == 20, "evaluate 20 random states"
    assert all(abs(np.imag(e)) < 1e-9 for e in random_energies), (
        "each energy <psi|H|psi> is real for a Hermitian H"
    )
    assert all(np.real(e) >= H2_FCI - 1e-9 for e in random_energies), (
        "the variational principle forbids any state from beating H2_FCI"
    )

## Summary

- A molecule becomes a **qubit Hamiltonian**: STO-3G H2 is `H2_TERMS`, a weighted sum of **15** Pauli strings on **4** qubits, big-endian (character `k` acts on qubit `k`).
- Its structure is what costs you on hardware: **15** terms to measure, **max locality 4** (the Jordan-Wigner doubles touch every qubit), and an `IIII` term that is just a constant energy shift.
- `hamiltonian_matrix(H2_TERMS)` is a $16\times16$ Hermitian matrix; its **lowest eigenvalue equals `H2_FCI`** = -1.137 Ha, the exact ground energy.
- The **Hartree-Fock determinant `|1100>`** (index `int("1100", 2)`) has energy `H2_HF` and, by the variational principle, sits *above* the ground state. A single Givens double excitation closes the gap exactly.
- The same physics survives **tapering** to one qubit, `H2_C0 * I + H2_CZ * Z + H2_CX * X`, whose ground energy `H2_C0 - hypot(H2_CZ, H2_CX)` is again `H2_FCI` — the naive qubit count is rarely the real one.
- The coefficients come from a classical **geometry -> integrals -> Jordan-Wigner** pipeline (`lib.chemistry.build_h2_hamiltonian`), shown as non-executed reference because it needs OpenFermion + PySCF.

**Next:** [`02-fermion-qubit-mapping.ipynb`](02-fermion-qubit-mapping.ipynb) — how the Jordan-Wigner Z-strings (and their Bravyi-Kitaev alternative) turn fermionic operators into the Pauli terms you just diagonalized.